# HMM Post-Processing

Random Forest + Hidden Markov Model for sleep state classification.

Adding HMM post processign since RF, XGBoost, and SVM classifies each 4 second epoch independently. 
The HMM applies Viterbi decoding to the RF probability outputs using the temporal transition structure of sleep states so it produces more realistic sequences

Uses PyEcog's `HMM_LL` class from `hmm_pyecog.py`.
 
`hmm_pyecog.py` is not written by me (Kyri) and was already within Marco's PyEcog branch.

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import glob
import os
import sys
import warnings
warnings.filterwarnings('ignore')
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    balanced_accuracy_score, cohen_kappa_score, matthews_corrcoef
)
sys.path.insert(0, os.path.abspath('..'))
from pyecog2.convert_figshare_sleep_data import readbinary_dat
from pyecog2.hmm_pyecog import HMM_LL
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 10

CLASSES=np.array(['n', 'r', 'w'])
CLASS_NAMES = {'n': 'NREM', 'r': 'REM', 'w': 'Wake'}

Leave-one-out CV with predict_proba

In [ ]:
def extract_power(power_spectra):
    bands = {
        'delta1':(0.75, 1.75),
        'delta2':(2.5, 3.5),
        'theta':(6.0, 9.0),
        'sigma':(10.0, 15.0),
        'beta_lowgamma':(18.0, 45.0)
    }
    result = {}
    for name, (lo, hi) in bands.items():
        result[name] = power_spectra[:, int(lo/0.25):int(hi/0.25)].sum(axis=1)
    return result

def load_data(dat_file):
    scores, power_spectra, eeg_var, emg_var, _ = readbinary_dat(dat_file)
    artifact_map = {'1': 'w', '2': 'n', '3':'r'}
    states = np.array([artifact_map.get(s, s) for s in scores])
    bands = extract_power(power_spectra)
    df = pd.DataFrame({'state': states, 'eeg_variance': eeg_var, 'emg_variance': emg_var, **bands})
    power_cols = ['delta1', 'delta2', 'theta', 'sigma', 'beta_lowgamma', 'eeg_variance','emg_variance']
    df[power_cols] = np.log(df[power_cols].clip(lower=1e-30))
    df['delta2_delta1_ratio'] = df['delta2']-df['delta1']
    return df[df['state'].isin(['w','n','r'])].reset_index(drop=True)

In [5]:
dat_files = sorted(glob.glob('data/M*EXP1.dat'))
mouse_dfs = {}
for dat_file in dat_files:
    mouse_id = os.path.splitext(os.path.basename(dat_file))[0]
    mouse_dfs[mouse_id] = load_data(dat_file)
print(f'Loaded {len(mouse_dfs)} mice')

Loaded 29 mice


In [7]:
FEATURES = ['delta1', 'delta2', 'theta', 'sigma', 'beta_lowgamma', 'eeg_variance',
            'emg_variance', 'delta2_delta1_ratio']
LABEL = 'state'

In [ ]:
mouse_ids = list(mouse_dfs.keys())
rf_results = []
per_mouse = {}
all_y_true_rf = []
all_y_pred_rf = []

for test_mouse in mouse_ids:
    train_mice = [m for m in mouse_ids if m != test_mouse]
    train_df = pd.concat([mouse_dfs[m] for m in train_mice], ignore_index=True)
    test_df  = mouse_dfs[test_mouse]

    X_train = train_df[FEATURES].values
    y_train = train_df[LABEL].values
    X_test = test_df[FEATURES].values
    y_test = test_df[LABEL].values

    clf = RandomForestClassifier(
        n_estimators=100, class_weight='balanced', n_jobs=-1, random_state=7
    )
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    proba = clf.predict_proba(X_test) 

    # making sure argmax of proba matches the predictions
    assert np.all(clf.classes_[np.argmax(proba, axis=1)] == y_pred), f'argmax does not match prediction for {test_mouse}'
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    rf_results.append({'mouse': test_mouse, 'balanced_accuracy': bal_acc})
    all_y_true_rf.extend(y_test)
    all_y_pred_rf.extend(y_pred)

    per_mouse[test_mouse] = {
        'y_true': y_test,
        'y_pred': y_pred,
        'proba': proba,
        'classes': clf.classes_,
        'y_train': y_train, #for transition matrix
        'train_mice': train_mice
    }
    print(f'{test_mouse}: RF balanced accuracy = {bal_acc:.3f}')

In [12]:
print(f'\nRF mean balanced accuracy: {rf_results_df["balanced_accuracy"].mean():.3f} +/- {rf_results_df["balanced_accuracy"].std():.3f}')


RF mean balanced accuracy: 0.797 +/- 0.121


In [ ]:
#cleared output so using per_mouse from memory
all_y_true_rf = []
all_y_pred_rf = []
for d in per_mouse.values():
    all_y_true_rf.extend(d['y_true'])
    all_y_pred_rf.extend(d['y_pred'])

In [17]:
kappa_rf = cohen_kappa_score(all_y_true_rf, all_y_pred_rf)
mcc_rf   = matthews_corrcoef(all_y_true_rf, all_y_pred_rf)
print(f"Cohen's kappa: {kappa_rf:.4f}  MCC: {mcc_rf:.4f}")

Cohen's kappa: 0.7921  MCC: 0.7925
